# JAX transformations for data scientists

**P2 Extension · D2 Independent · 70 minutes · not assessed**

This compact bridge covers arrays, pure functions, explicit random keys, automatic differentiation, compilation, and vectorisation. It deliberately excludes neural network construction, PyTrees, Optax, accelerators, and long training loops.

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np

## Pure numerical function

JAX transformations operate most predictably on functions whose outputs depend only on explicit inputs. The small linear mean-squared error is our complete oracle.

In [ ]:
def mse_loss(theta, x, y):
    prediction = x @ theta
    return jnp.mean((prediction - y) ** 2)

x = jnp.array([[1.0, -1.0], [1.0, 0.0], [1.0, 2.0], [1.0, 3.0]])
true_theta = jnp.array([0.5, 1.25])

## Explicit randomness

A random key is data: split it and pass the subkey that owns each random operation. Reusing a key repeats the same values and is usually an error.

In [ ]:
def noisy_targets(key, x, theta, scale=0.05):
    key, noise_key = jax.random.split(key)
    noise = scale * jax.random.normal(noise_key, shape=(x.shape[0],))
    return key, x @ theta + noise

key = jax.random.key(20260718)
key, y = noisy_targets(key, x, true_theta)
assert y.shape == (4,)
assert jnp.isfinite(y).all()

## Differentiate and verify

Automatic differentiation is executable chain-rule bookkeeping, not independent evidence that a model or implementation is correct. Compare it with a finite-difference oracle on a small deterministic input.

In [ ]:
def central_difference(function, theta, step=1e-3):
    basis = jnp.eye(theta.size, dtype=theta.dtype)
    return jax.vmap(
        lambda direction: (function(theta + step * direction)
                           - function(theta - step * direction)) / (2 * step)
    )(basis)

theta = jnp.array([0.0, 0.0])
value, gradient = jax.value_and_grad(mse_loss)(theta, x, y)
finite_difference = central_difference(lambda t: mse_loss(t, x, y), theta)
assert jnp.isfinite(value)
assert jnp.allclose(gradient, finite_difference, atol=2e-3, rtol=2e-3)

## Compile and vectorise

`jit` specialises a pure function for input shapes and dtypes. The first call includes compilation, so a one-call wall-clock comparison is misleading. `vmap` introduces a batch axis without writing a Python loop.

In [ ]:
compiled_gradient = jax.jit(jax.grad(mse_loss))
compiled_value = compiled_gradient(theta, x, y)

def example_loss(theta, row, target):
    return (row @ theta - target) ** 2

per_example_loss = jax.vmap(example_loss, in_axes=(None, 0, 0))
per_example_gradient = jax.vmap(jax.grad(example_loss), in_axes=(None, 0, 0))
losses = per_example_loss(theta, x, y)
gradients = per_example_gradient(theta, x, y)
assert jnp.allclose(losses.mean(), mse_loss(theta, x, y))
assert jnp.allclose(gradients.mean(axis=0), gradient)
assert jnp.allclose(compiled_value, gradient)
{'loss': float(value), 'gradient': np.asarray(gradient), 'batch_shape': gradients.shape}

## Transfer and boundary

Replace mean-squared error with the stable binary logistic loss from the core course. Verify one gradient, then compare scalar and vectorised per-example losses. Record shape, dtype, key ownership, and the compilation boundary. Continue with model construction, PyTrees, Optax, accelerators, and physics-informed objectives in Scientific Machine Learning—not in the required Data Processing assessment path.